# Production Inference Pipeline
Loads the `.joblib` model weights generated by `training.ipynb` and runs the full hybrid forecast on incoming data.

**Pipeline:** KNN Imputation → IQR + Isolation Forest Anomaly Imputation → Prophet Baseline → LightGBM Residual Correction

**Key Principle:** Both Prophet and LightGBM were trained strictly on training data only. Val and test were never exposed during training — ensuring zero data leakage.

## 1. Setup & Model Loading

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

warnings.filterwarnings('ignore')

MODELS_DIR = '../Models'
OUTPUT_DIR = '../Outputs'

TARGET_COL = 'Demand_MWh'

# Must match the features used during training in hybrid_model.py
FEATURE_CANDIDATES = [
    'Day_of_Week', 'Is_Weekend', 'Is_Holiday',
    'Month', 'DayOfYear', 'WeekOfYear', 'Trend',
    'Avg_Temp', 'Rainfall', 'Temp_Lag_1',
    'Lag_1', 'Lag_2', 'Lag_7', 'Lag_14', 'Lag_30',
    'Rolling_7', 'Rolling_14', 'Rolling_30',
]

In [ ]:
# Load serialized models
try:
    prophet_model = joblib.load(os.path.join(MODELS_DIR, 'prophet_model.joblib'))
    lgbm_model = joblib.load(os.path.join(MODELS_DIR, 'lgbm_model.joblib'))
    iso_forest = joblib.load(os.path.join(MODELS_DIR, 'iso_forest.joblib'))
    knn_imputer = joblib.load(os.path.join(MODELS_DIR, 'knn_imputer.joblib'))
    print('SUCCESS: Model weights and Imputers loaded.')
except FileNotFoundError:
    raise RuntimeError('Models not found. Run training.ipynb first.')

## 2. Data Ingestion & Pre-processing

In [ ]:
# Load incoming data (in production, replace with live data source)
df_infer = pd.read_csv(os.path.join('../test_data', 'dataset_daily_test.csv'))
df_infer['Date'] = pd.to_datetime(df_infer['Date'])

# KNN imputation for missing values (No Leakage: Imputer was fitted on original train data)
df_infer['Time_Idx'] = df_infer['Date'].dt.dayofyear
features_to_impute = ['Time_Idx', 'Demand_MWh', 'Avg_Temp', 'Rainfall']
for c in features_to_impute:
    if c not in df_infer.columns:
        df_infer[c] = np.nan
df_infer[features_to_impute] = knn_imputer.transform(df_infer[features_to_impute])
df_infer.drop(columns=['Time_Idx'], inplace=True)

# Feature engineering (must match training pipeline)
df_infer['Month']      = df_infer['Date'].dt.month
df_infer['DayOfYear']  = df_infer['Date'].dt.dayofyear
df_infer['WeekOfYear'] = df_infer['Date'].dt.isocalendar().week.astype(int)
df_infer['Trend']      = (df_infer['Date'] - pd.Timestamp('2018-01-01')).dt.days
df_infer['Lag_2']      = df_infer[TARGET_COL].shift(2)
df_infer['Lag_14']     = df_infer[TARGET_COL].shift(14)
df_infer['Rolling_14'] = df_infer[TARGET_COL].rolling(window=14, min_periods=1).mean()
df_infer['Rolling_30'] = df_infer[TARGET_COL].rolling(window=30, min_periods=1).mean()
df_infer['Temp_Lag_1'] = df_infer['Avg_Temp'].shift(1)

features = [c for c in FEATURE_CANDIDATES if c in df_infer.columns]
df_clean = df_infer.dropna(subset=features).copy()
print(f'Loaded {len(df_clean)} rows with {len(features)} features.')

## 3. Anomaly Detection & Imputation

In [ ]:
if_predictions = iso_forest.predict(df_clean[features])

if TARGET_COL in df_clean.columns:
    q1 = df_clean[TARGET_COL].quantile(0.25)
    q3 = df_clean[TARGET_COL].quantile(0.75)
    iqr_val = q3 - q1
    iqr_anomalies = np.where(
        (df_clean[TARGET_COL] < q1 - 1.5 * iqr_val) | (df_clean[TARGET_COL] > q3 + 1.5 * iqr_val), -1, 1)
    is_anomaly = (if_predictions == -1) | (iqr_anomalies == -1)
    for idx in np.where(is_anomaly)[0]:
        start = max(0, idx - 7)
        clean_mask = ~is_anomaly[start:idx]
        clean_window = df_clean[TARGET_COL].iloc[start:idx][clean_mask]
        df_clean.iloc[idx, df_clean.columns.get_loc(TARGET_COL)] = clean_window.mean() if len(clean_window) > 0 else df_clean[TARGET_COL].mean()
    print(f'Imputed {is_anomaly.sum()} anomalies via IQR + Isolation Forest.')
else:
    is_anomaly = if_predictions == -1
    print(f'No target column — flagged {is_anomaly.sum()} anomalies via Isolation Forest only.')

df_clean['Anomaly_Flag'] = np.where(is_anomaly, 'ALERT', 'OK')

## 4. Hybrid Forecast

In [ ]:
# Prophet baseline (with temperature regressor)
df_prophet = df_clean[['Date', 'Avg_Temp']].rename(columns={'Date': 'ds'})
prophet_preds = prophet_model.predict(df_prophet)['yhat'].values

# LightGBM residual corrections
lgbm_preds = lgbm_model.predict(df_clean[features])

# Final hybrid combination
df_clean['Prophet_Pred'] = prophet_preds
df_clean['LGBM_Correction'] = lgbm_preds
df_clean['Forecast_MWh'] = prophet_preds + lgbm_preds

print('--- INFERENCE PREVIEW ---')
display(df_clean[['Date', 'Demand_MWh', 'Prophet_Pred', 'LGBM_Correction', 'Forecast_MWh', 'Anomaly_Flag']].tail(10))

## 5. Explainability — SHAP Values (Explainable AI)

### Apa itu SHAP?
**SHAP (SHapley Additive exPlanations)** adalah metode matematika dari teori permainan kooperatif yang menghitung **kontribusi** setiap fitur terhadap setiap prediksi individual. SHAP menjawab pertanyaan: *"Seberapa besar fitur X mendorong prediksi naik atau turun dari rata-rata?"*

### Mengapa Perlu SHAP?
Model LightGBM adalah *black-box* — ia menghasilkan prediksi tanpa menjelaskan logikanya. SHAP membongkar *black-box* ini dengan memberikan **transparansi penuh** atas setiap keputusan model. Ini krusial untuk:
1. **Audit regulasi**: Menjelaskan kepada regulator mengapa model memprediksi lonjakan/penurunan demand.
2. **Deteksi drift**: Jika fitur yang paling berpengaruh berubah drastis, ini sinyal bahwa data baru berbeda dari data latih.
3. **Kepercayaan stakeholder**: Operator pembangkit listrik dapat memvalidasi apakah alasan AI masuk akal secara domain.

### Cara Membaca Hasil SHAP
- **SHAP Value Positif (+)**: Fitur ini **meningkatkan** prediksi demand di atas rata-rata.
- **SHAP Value Negatif (-)**: Fitur ini **menurunkan** prediksi demand di bawah rata-rata.
- **Besarnya (|SHAP|)**: Semakin besar nilai absolut, semakin kuat pengaruh fitur tersebut.

### 5a. Global SHAP Summary (Seluruh Dataset)
Menunjukkan **fitur mana yang paling berpengaruh secara keseluruhan** di semua data inferensi.

**Cara membaca Summary Plot:**
- Setiap titik = satu data point (satu hari).
- Sumbu X = SHAP value (dampak ke prediksi dalam MWh).
- Warna **merah** = nilai fitur tinggi, **biru** = nilai fitur rendah.
- Fitur diurutkan dari atas (paling berpengaruh) ke bawah (paling lemah).

In [ ]:
print('Menghitung SHAP values untuk seluruh dataset inferensi...')
explainer = shap.TreeExplainer(lgbm_model)
shap_vals = explainer.shap_values(df_clean[features])

print('\n--- GLOBAL SHAP SUMMARY ---')
print('Fitur di atas paling berpengaruh. Merah = nilai tinggi, Biru = nilai rendah.')
shap.summary_plot(shap_vals, df_clean[features], feature_names=features, show=True)

### 5b. Korelasi Fitur-SHAP: Apa Arti Setiap Fitur?
Tabel di bawah menerjemahkan *summary plot* menjadi **narasi bisnis** yang dapat dipahami oleh non-teknis.

| Fitur | Korelasi dengan Demand | Interpretasi Bisnis |
|-------|------------------------|---------------------|
| **Lag_1** | Positif kuat | Demand kemarin tinggi → demand hari ini kemungkinan juga tinggi (inersia pola konsumsi) |
| **Rolling_7** | Positif kuat | Rata-rata demand 7 hari terakhir naik → tren mingguan meningkat |
| **Avg_Temp** | Bisa dua arah | Suhu ekstrem (sangat panas/dingin) → AC/pemanas → demand naik |
| **DayOfYear** | Musiman | Bulan tertentu (musim kemarau/hujan) memiliki pola demand berbeda |
| **Is_Weekend** | Negatif | Akhir pekan → industri tutup → demand turun |
| **Is_Holiday** | Negatif | Hari libur → pabrik berhenti → demand turun |
| **Trend** | Positif | Demand listrik tumbuh seiring waktu (pertumbuhan ekonomi) |
| **Rainfall** | Negatif ringan | Hujan → udara lebih sejuk → AC kurang digunakan → demand sedikit turun |

### 5c. Mean Absolute SHAP (Feature Importance Ranking)
Menunjukkan **rata-rata dampak absolut** setiap fitur. Fitur di atas memiliki pengaruh paling besar ke prediksi (terlepas dari arah positif/negatif).

In [ ]:
# Mean |SHAP| importance
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
importance_df = pd.DataFrame({'Feature': features, 'Mean |SHAP| (MWh)': mean_abs_shap})
importance_df = importance_df.sort_values('Mean |SHAP| (MWh)', ascending=False).reset_index(drop=True)

print('--- FEATURE IMPORTANCE (Mean |SHAP|) ---')
display(importance_df)

shap.summary_plot(shap_vals, df_clean[features], feature_names=features, plot_type='bar', show=True)

### 5d. Narasi Prediksi Terakhir (Single-Point Explanation)
Menjelaskan **mengapa AI memprediksi angka tertentu** pada hari terakhir data.

In [ ]:
print('\n--- EXPLAINABILITY: DATA POINT TERAKHIR ---')

latest_shap = shap_vals[-1]
latest_input = df_clean[features].iloc[-1]
latest_date  = df_clean['Date'].iloc[-1]

feature_impacts = list(zip(features, latest_shap, latest_input.values))
feature_impacts.sort(key=lambda x: abs(x[1]), reverse=True)

print(f'\nTanggal inferensi terakhir: {latest_date:%Y-%m-%d}')
print(f'Prediksi akhir (Hybrid): {df_clean["Forecast_MWh"].iloc[-1]:,.0f} MWh')
if TARGET_COL in df_clean.columns:
    actual = df_clean[TARGET_COL].iloc[-1]
    error_pct = abs(df_clean['Forecast_MWh'].iloc[-1] - actual) / actual * 100
    print(f'Demand aktual: {actual:,.0f} MWh (error: {error_pct:.2f}%)')

print(f'\nTop-5 Faktor Penentu Keputusan AI:')
print('=' * 75)
for i, (feat, impact, val) in enumerate(feature_impacts[:5]):
    direction = '📈 MENINGKATKAN prediksi' if impact > 0 else '📉 MENURUNKAN prediksi'
    print(f'{i+1}. {feat}')
    print(f'   Nilai fitur hari ini : {val:.2f}')
    print(f'   Arah dampak          : {direction}')
    print(f'   Besar dampak         : {abs(impact):,.2f} MWh')
    print()
print('=' * 75)

### 5e. Waterfall Plot (Visualisasi Dekomposisi Prediksi)
Menunjukkan bagaimana prediksi dibangun langkah demi langkah dari *base value* (rata-rata model) hingga prediksi akhir. Setiap bar menunjukkan kontribusi satu fitur.

In [ ]:
# Waterfall plot for the latest prediction
explanation = shap.Explanation(
    values=shap_vals[-1],
    base_values=explainer.expected_value,
    data=df_clean[features].iloc[-1].values,
    feature_names=features
)
print(f'Waterfall: Dekomposisi prediksi untuk {latest_date:%Y-%m-%d}')
shap.waterfall_plot(explanation, show=True)

### 5f. Force Plot (Visualisasi Gaya Tarik-Menarik Fitur)
Menunjukkan *"tarik-menarik"* antar fitur. Merah mendorong prediksi **naik**, biru mendorong prediksi **turun**. Lebar bar = kekuatan pengaruh.

In [ ]:
# Force plot for latest prediction
shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_vals[-1],
    df_clean[features].iloc[-1],
    feature_names=features,
    matplotlib=True
)
plt.tight_layout()
plt.show()

### 5g. Dependence Plot (Korelasi Fitur ↔ Dampak SHAP)
Menunjukkan **hubungan non-linear** antara nilai fitur dan dampaknya ke prediksi. Scatter plot ini mengungkap interaksi tersembunyi antar fitur.

Contoh interpretasi:
- Jika **Avg_Temp** menunjukkan pola U-shaped: suhu rendah DAN tinggi keduanya meningkatkan demand (pemanas & AC).
- Jika **Lag_1** menunjukkan garis lurus naik: semakin tinggi demand kemarin, semakin tinggi pula prediksi hari ini (inersia).

In [ ]:
# Dependence plots for top-3 features
top_features = importance_df['Feature'].head(3).tolist()
for feat in top_features:
    print(f'\nDependence Plot: {feat}')
    shap.dependence_plot(feat, shap_vals, df_clean[features], feature_names=features, show=True)

### 5h. Narasi Lengkap (Auto-Generated)
Ringkasan yang dapat langsung di-copy ke laporan atau dashboard.

In [ ]:
print('=' * 75)
print('NARASI OTOMATIS UNTUK LAPORAN')
print('=' * 75)

# Global narrative
top3_global = importance_df.head(3)
print(f'\n[GLOBAL] Secara keseluruhan, tiga fitur paling berpengaruh terhadap')
print(f'koreksi residual LightGBM pada {len(df_clean)} hari data inferensi adalah:')
for _, row in top3_global.iterrows():
    print(f"  • {row['Feature']} (rata-rata dampak: {row['Mean |SHAP| (MWh)']:,.2f} MWh per hari)")

# Latest point narrative
parts = []
for feat, impact, val in feature_impacts[:3]:
    arah = 'meningkatkan' if impact > 0 else 'menurunkan'
    parts.append(f'{feat} (nilai={val:.2f}) {arah} prediksi sebesar {abs(impact):,.2f} MWh')

narasi = (
    f'\n[TERBARU] Pada {latest_date:%Y-%m-%d}, tiga faktor utama yang mempengaruhi '
    f'prediksi demand listrik adalah: {parts[0]}; {parts[1]}; '
    f'dan {parts[2]}.'
)
print(narasi)
print('\n' + '=' * 75)

## 6. Export Results

In [ ]:
output_path = os.path.join(OUTPUT_DIR, 'inference_results.csv')
df_clean.to_csv(output_path, index=False)
print(f'\nInference complete. Output saved to: {output_path}')